# AccessAI - Prototipo de detección urbana

Notebook local para Windows 11 y NVIDIA CUDA. Prepara el ROD-Dataset con el mapeo original de 25 clases a las cuatro categorías del proyecto, entrena YOLO26n, evalúa el modelo y permite predecir imágenes de una carpeta local.

## Clases finales

| ID | Clase |
|---:|---|
| 0 | `Obstaculo_Dinamico` |
| 1 | `Obstaculo_Fijo` |
| 2 | `Barrera_Arquitectonica` |
| 3 | `Infraestructura_Peatonal` |

El notebook no inicia entrenamiento si PyTorch no detecta CUDA. Los resultados se guardan en una carpeta única bajo `runs/detect/`.

# 01. Configuración

In [2]:
import gc
import math
import platform
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)
print("ok" if torch.cuda.is_available() else "no ok")       
if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(gpu_properties.total_memory / 1024**3, 2), "GB")
else:
    print("GPU: no detectada por CUDA")

Python: 3.11.16 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:36:16) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.14.0+cu126
CUDA de PyTorch: 12.6
CUDA disponible: True
Ultralytics: 8.4.69
ok
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.0 GB


# 02. Diagnóstico del sistema

In [3]:
print("Sistema operativo:", platform.platform())
print("Carpeta actual del kernel:", Path.cwd())
print("Entorno Python activo:", Path(sys.prefix).name)
if platform.system() != "Windows":
    print("ADVERTENCIA: este notebook está preparado para Windows 11.")
if sys.version_info[:3] != (3, 11, 16):
    print("ADVERTENCIA: la versión objetivo de Python es 3.11.16.")
if torch.__version__ != "2.14.0+cu126":
    print("ADVERTENCIA: la versión objetivo de PyTorch es 2.14.0+cu126.")
if not ultralytics.__version__.startswith("8.4."):
    print("ADVERTENCIA: se espera Ultralytics de la serie 8.4.x.")
if Path(sys.prefix).name != "proyecto_yolo":
    print("ADVERTENCIA: selecciona el kernel Conda 'proyecto_yolo' antes de ejecutar el notebook.")

Sistema operativo: Windows-10-10.0.22631-SP0
Carpeta actual del kernel: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2
Entorno Python activo: proyecto_yolo


# 03. Diagnóstico NVIDIA CUDA

In [4]:
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = None
GPU_NAME = "No disponible"
GPU_VRAM_GB = 0.0
FREE_VRAM_GB = 0.0

if CUDA_AVAILABLE:
    DEVICE = 0
    GPU_NAME = torch.cuda.get_device_name(0)
    gpu_properties = torch.cuda.get_device_properties(0)
    GPU_VRAM_GB = gpu_properties.total_memory / 1024**3
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    FREE_VRAM_GB = free_bytes / 1024**3

    print("PyTorch:", torch.__version__)
    print("CUDA compilado:", torch.version.cuda)
    print("CUDA disponible:", torch.cuda.is_available())
    print("GPU:", GPU_NAME)
    print("VRAM:", round(GPU_VRAM_GB, 2), "GB")
    print("VRAM libre:", round(FREE_VRAM_GB, 2), "GB")

    if "NVIDIA" not in GPU_NAME.upper():
        CUDA_AVAILABLE = False
        DEVICE = None
        print("ADVERTENCIA: no se detectó una GPU NVIDIA; entrenamiento detenido.")
else:
    print("PyTorch:", torch.__version__)
    print("CUDA compilado:", torch.version.cuda)
    print("CUDA disponible:", torch.cuda.is_available())
    print("ADVERTENCIA: CUDA no está disponible. El entrenamiento queda detenido.")
    print("Soluciona primero el controlador NVIDIA y PyTorch con CUDA.")

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    nvidia_report = subprocess.run(
        [nvidia_smi], capture_output=True, text=True, check=False
    )
    print("\nDiagnóstico nvidia-smi:")
    print(nvidia_report.stdout if nvidia_report.stdout else nvidia_report.stderr)
else:
    print("nvidia-smi no está disponible; se omite el diagnóstico opcional.")

PyTorch: 2.14.0+cu126
CUDA compilado: 12.6
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.0 GB
VRAM libre: 6.93 GB

Diagnóstico nvidia-smi:
Thu Oct  1 15:30:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P0             21W /   80W |      91MiB /   8188MiB | 

# 04. Rutas del proyecto

In [5]:
ROOT = Path.cwd()
DATASET_PATH = ROOT / "DATA" / "ROD-Dataset" / "dataset"
YAML_ORIGINAL_PATH = DATASET_PATH / "data.yaml"
YAML_FILTERED_PATH = DATASET_PATH / "data_filtrado.yaml"
MODEL_PATH = ROOT / "yolo26n.pt"
RUNS_DIR = ROOT / "runs" / "detect"

print("Raíz del proyecto:", ROOT)
print("Dataset:", DATASET_PATH)
print("Modelo inicial:", MODEL_PATH)

if not (ROOT / "DATA").is_dir():
    raise FileNotFoundError(
        f"No se encuentra la carpeta DATA en {ROOT}. "
        "Abre el notebook desde la raíz del proyecto AccessAI."
    )
if not DATASET_PATH.is_dir():
    raise FileNotFoundError(f"No existe el dataset esperado:\n{DATASET_PATH}")

Raíz del proyecto: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2
Dataset: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset
Modelo inicial: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\yolo26n.pt


# 05. Dataset

In [6]:
if not YAML_ORIGINAL_PATH.is_file():
    raise FileNotFoundError(f"No existe el data.yaml original:\n{YAML_ORIGINAL_PATH}")

with YAML_ORIGINAL_PATH.open("r", encoding="utf-8") as yaml_file:
    CONFIG_ORIGINAL = yaml.safe_load(yaml_file)

for split_name, split_path in (
    ("train", DATASET_PATH / "train" / "images"),
    ("valid", DATASET_PATH / "valid" / "images"),
    ("test", DATASET_PATH / "test" / "images"),
):
    if not split_path.is_dir():
        raise FileNotFoundError(
            f"No se encuentra la carpeta de imágenes del split {split_name}:\n{split_path}"
        )

print("data.yaml original:", YAML_ORIGINAL_PATH)
print("Clases originales declaradas:", CONFIG_ORIGINAL.get("nc"))
print("Splits train/valid/test: encontrados")

data.yaml original: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data.yaml
Clases originales declaradas: 25
Splits train/valid/test: encontrados


# 06. Verificación de las 4 clases

In [7]:
MAPEO_CLASES = {
    0: 0, 2: 0, 3: 0, 8: 0, 10: 0, 15: 0, 16: 0,
    5: 1, 6: 1, 9: 1, 12: 1, 13: 1, 17: 1, 18: 1,
    19: 1, 20: 1, 21: 1, 22: 1, 23: 1, 24: 1,
    4: 2,
    7: 3, 11: 3, 14: 3,
}

NUEVOS_NOMBRES = [
    "Obstaculo_Dinamico",
    "Obstaculo_Fijo",
    "Barrera_Arquitectonica",
    "Infraestructura_Peatonal",
]
NUM_CLASSES = 4

if CONFIG_ORIGINAL.get("nc") != 25:
    raise ValueError(
        "El data.yaml original no declara las 25 clases esperadas por el mapeo actual. "
        "Revisa el dataset antes de continuar."
    )
if len(NUEVOS_NOMBRES) != NUM_CLASSES:
    raise ValueError("La configuración debe conservar exactamente las cuatro clases del proyecto.")

for class_id, class_name in enumerate(NUEVOS_NOMBRES):
    print(f"{class_id}: {class_name}")

CONFIG_FILTRADA = {
    "path": str(DATASET_PATH.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NUM_CLASSES,
    "names": NUEVOS_NOMBRES,
}
with YAML_FILTERED_PATH.open("w", encoding="utf-8") as yaml_file:
    yaml.safe_dump(CONFIG_FILTRADA, yaml_file, sort_keys=False, allow_unicode=True)

with YAML_FILTERED_PATH.open("r", encoding="utf-8") as yaml_file:
    CONFIG_FILTRADA = yaml.safe_load(yaml_file)

if CONFIG_FILTRADA.get("nc") != 4:
    raise ValueError("data_filtrado.yaml debe declarar nc: 4.")
if CONFIG_FILTRADA.get("names") != NUEVOS_NOMBRES:
    raise ValueError("Las clases de data_filtrado.yaml no coinciden con las clases del proyecto.")

print("\nYAML de entrenamiento:", YAML_FILTERED_PATH)
print("nc:", CONFIG_FILTRADA["nc"])
print("names:", CONFIG_FILTRADA["names"])

0: Obstaculo_Dinamico
1: Obstaculo_Fijo
2: Barrera_Arquitectonica
3: Infraestructura_Peatonal

YAML de entrenamiento: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data_filtrado.yaml
nc: 4
names: ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']


# 07. Comprobación de imágenes y etiquetas

In [9]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def preparar_etiquetas(split_name):
    split_dir = DATASET_PATH / split_name
    active_labels_dir = split_dir / "labels"
    original_labels_dir = split_dir / "labels_originales"
    temporary_labels_dir = split_dir / "labels_filtradas_tmp"

    if not split_dir.is_dir():
        raise FileNotFoundError(f"No existe el split {split_name}:\n{split_dir}")
    if original_labels_dir.is_dir():
        source_dir = original_labels_dir
    elif active_labels_dir.is_dir():
        shutil.copytree(active_labels_dir, original_labels_dir)
        source_dir = original_labels_dir
        print(f"{split_name}: copia de seguridad original creada.")
    else:
        raise FileNotFoundError(
            f"No se encuentran etiquetas en {split_dir}. Se esperaba 'labels' o 'labels_originales'."
        )

    if temporary_labels_dir.exists():
        shutil.rmtree(temporary_labels_dir)
    temporary_labels_dir.mkdir(parents=True)

    class_counts = Counter()
    discarded_counts = Counter()
    for label_file in sorted(source_dir.glob("*.txt")):
        converted_lines = []
        with label_file.open("r", encoding="utf-8") as source_file:
            for line_number, line in enumerate(source_file, start=1):
                parts = line.strip().split()
                if not parts:
                    continue
                is_box = len(parts) == 5
                is_segment = len(parts) >= 7 and (len(parts) - 1) % 2 == 0
                if not is_box and not is_segment:
                    raise ValueError(
                        f"Etiqueta incorrecta en {label_file}, línea {line_number}: "
                        "se esperaba una caja YOLO (5 campos) o un polígono "
                        "(clase y al menos 3 pares x/y)."
                    )
                try:
                    original_class = int(parts[0])
                    box_values = [float(value) for value in parts[1:]]
                except ValueError as error:
                    raise ValueError(
                        f"Etiqueta no numérica en {label_file}, línea {line_number}."
                    ) from error
                if not all(math.isfinite(value) for value in box_values):
                    raise ValueError(
                        f"Coordenada no finita en {label_file}, línea {line_number}."
                    )
                if any(value < 0.0 or value > 1.0 for value in box_values):
                    raise ValueError(
                        f"Coordenadas fuera del rango 0..1 en {label_file}, línea {line_number}."
                    )
                if is_box and (box_values[2] == 0.0 or box_values[3] == 0.0):
                    raise ValueError(
                        f"Caja sin ancho o alto en {label_file}, línea {line_number}."
                    )
                if original_class not in MAPEO_CLASES:
                    if original_class < 0 or original_class >= 25:
                        raise ValueError(
                            f"ID de clase {original_class} no válido en {label_file}, línea {line_number}."
                        )
                    discarded_counts[original_class] += 1
                    continue

                new_class = MAPEO_CLASES[original_class]
                parts[0] = str(new_class)
                converted_lines.append(" ".join(parts))
                class_counts[new_class] += 1

        output_file = temporary_labels_dir / label_file.name
        output_file.write_text(
            "\n".join(converted_lines) + ("\n" if converted_lines else ""),
            encoding="utf-8",
        )

    if active_labels_dir.exists():
        shutil.rmtree(active_labels_dir)
    temporary_labels_dir.replace(active_labels_dir)

    images_dir = split_dir / "images"
    image_count = sum(
        1 for path in images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    label_count = sum(1 for path in active_labels_dir.glob("*.txt"))
    print(f"{split_name}: {image_count} imágenes, {label_count} archivos de etiquetas.")
    print("  Objetos por clase:", dict(sorted(class_counts.items())))
    print("  Clases originales descartadas:", dict(sorted(discarded_counts.items())))


for split_name in ("train", "valid", "test"):
    preparar_etiquetas(split_name)

print("Preparación terminada. Las etiquetas originales permanecen en labels_originales.")

train: 19186 imágenes, 19186 archivos de etiquetas.
  Objetos por clase: {0: 15411, 1: 10614, 2: 1125, 3: 4215}
  Clases originales descartadas: {1: 886}
valid: 3511 imágenes, 3511 archivos de etiquetas.
  Objetos por clase: {0: 2587, 1: 1552, 2: 339, 3: 993}
  Clases originales descartadas: {1: 101}
test: 1629 imágenes, 1629 archivos de etiquetas.
  Objetos por clase: {0: 781, 1: 846, 2: 223, 3: 468}
  Clases originales descartadas: {1: 54}
Preparación terminada. Las etiquetas originales permanecen en labels_originales.


# 08. Modelo YOLO26n

In [10]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"No se encuentra yolo26n.pt en:\n{MODEL_PATH}\n"
        "Coloca el checkpoint YOLO26n en la raíz del proyecto."
    )

model = YOLO("yolo26n.pt")
print("Modelo cargado:", MODEL_PATH)
print("Clases del dataset:", NUEVOS_NOMBRES)

Modelo cargado: c:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\yolo26n.pt
Clases del dataset: ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']


# 09. Configuración de entrenamiento

En Jupyter sobre Windows se usa `workers=0` para evitar procesos de datos duplicados o problemas de arranque con multiprocessing. Puede reducir la velocidad de carga frente a `workers=4`, pero prioriza estabilidad.

In [11]:
EPOCHS = 40
PATIENCE = 10
IMGSZ = 640
BATCH = 8
WORKERS = 0
AMP = True
CACHE = False
CLOSE_MOSAIC = 10
OPTIMIZER = "AdamW"
LR0 = 0.001
LRF = 0.01
SEED = 42

print("Configuración YOLO26n para RTX 4060 Laptop 8 GB")
print(f"Épocas={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, workers={WORKERS}")
print(f"AMP={AMP}, cache={CACHE}, patience={PATIENCE}, close_mosaic={CLOSE_MOSAIC}")
print(f"optimizer={OPTIMIZER}, lr0={LR0}, lrf={LRF}, seed={SEED}")
print("Clases:", NUEVOS_NOMBRES)

Configuración YOLO26n para RTX 4060 Laptop 8 GB
Épocas=40, imgsz=640, batch=8, workers=0
AMP=True, cache=False, patience=10, close_mosaic=10
optimizer=AdamW, lr0=0.001, lrf=0.01, seed=42
Clases: ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']


# 10. Entrenamiento

In [ ]:
if not CUDA_AVAILABLE or not torch.cuda.is_available() or DEVICE != 0:
    raise RuntimeError(
        "Entrenamiento detenido: CUDA y una GPU NVIDIA deben estar disponibles. "
        "No se iniciará entrenamiento en CPU. Revisa primero el controlador NVIDIA, "
        "PyTorch con CUDA y el kernel Conda proyecto_yolo."
    )
if "NVIDIA" not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError("Entrenamiento detenido: la GPU activa no se identifica como NVIDIA.")
if not YAML_FILTERED_PATH.is_file():
    raise FileNotFoundError(f"No existe el YAML de entrenamiento:\n{YAML_FILTERED_PATH}")
if CONFIG_FILTRADA.get("nc") != 4 or CONFIG_FILTRADA.get("names") != NUEVOS_NOMBRES:
    raise ValueError("Entrenamiento detenido: data_filtrado.yaml no contiene las 4 clases esperadas.")
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"No existe el checkpoint YOLO26n:\n{MODEL_PATH}")

free_bytes, total_bytes = torch.cuda.mem_get_info(0)
FREE_VRAM_GB = free_bytes / 1024**3
if FREE_VRAM_GB < 2.0:
    raise RuntimeError(
        f"VRAM insuficiente para comenzar con seguridad: quedan {FREE_VRAM_GB:.2f} GB libres. "
        "Cierra aplicaciones que usen la GPU y vuelve a ejecutar esta celda."
    )

RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "AccessAI_YOLO26n_4clases_RTX4060_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")

gc.collect()
torch.cuda.empty_cache()

try:
    train_results = model.train(
        data=str(YAML_FILTERED_PATH),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        amp=AMP,
        cache=CACHE,
        patience=PATIENCE,
        close_mosaic=CLOSE_MOSAIC,
        optimizer=OPTIMIZER,
        lr0=LR0,
        lrf=LRF,
        seed=SEED,
        deterministic=True,
        val=True,
        save=True,
        save_period=10,
        plots=True,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=False,
        verbose=True,
    )
except torch.cuda.OutOfMemoryError as error:
    gc.collect()
    torch.cuda.empty_cache()
    raise RuntimeError(
        "CUDA out of memory: el entrenamiento se ha detenido, sin cambiar a CPU. "
        "Cierra aplicaciones GPU y vuelve a intentarlo con BATCH=4."
    ) from error

RUN_DIR = Path(train_results.save_dir)
BEST_MODEL = RUN_DIR / "weights" / "best.pt"
LAST_MODEL = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

if not BEST_MODEL.is_file() or not LAST_MODEL.is_file():
    raise FileNotFoundError(f"No se generaron best.pt y last.pt en:\n{RUN_DIR}")

print("Entrenamiento terminado.")
print("Run:", RUN_DIR)
print("Best model:", BEST_MODEL)
print("Last model:", LAST_MODEL)
print("Resultados:", RESULTS_CSV)

del model
gc.collect()
torch.cuda.empty_cache()

New https://pypi.org/project/ultralytics/8.4.171 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.69  Python-3.11.16 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\III DIPLOMA DE EXTENSIN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data_filtrado.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0,

# 11. Validación

In [ ]:
if not BEST_MODEL.is_file():
    raise FileNotFoundError(f"No se encuentra el mejor modelo entrenado:\n{BEST_MODEL}")

trained_model = YOLO(str(BEST_MODEL))
EVAL_DIR = RUN_DIR / "test_evaluation"

try:
    metrics = trained_model.val(
        data=str(YAML_FILTERED_PATH),
        split="test",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=True,
        project=str(RUN_DIR),
        name="test_evaluation",
        exist_ok=False,
        verbose=True,
    )
except torch.cuda.OutOfMemoryError as error:
    gc.collect()
    torch.cuda.empty_cache()
    raise RuntimeError(
        "CUDA out of memory durante la evaluación de test. "
        "Reduce eval batch a 4 y vuelve a ejecutar la celda de validación."
    ) from error

print("Evaluación de test guardada en:", EVAL_DIR)
print("Modelo evaluado:", BEST_MODEL)

# 12. Métricas

In [ ]:
PRECISION = float(metrics.box.mp)
RECALL = float(metrics.box.mr)
MAP50 = float(metrics.box.map50)
MAP50_95 = float(metrics.box.map)

print("=" * 60)
print("MÉTRICAS GLOBALES EN TEST")
print("=" * 60)
print(f"Precision: {PRECISION:.4f}")
print(f"Recall:    {RECALL:.4f}")
print(f"mAP50:     {MAP50:.4f}")
print(f"mAP50-95:  {MAP50_95:.4f}")

class_precision = metrics.box.p
class_recall = metrics.box.r
class_map50 = metrics.box.ap50
class_map = metrics.box.ap
class_indices = metrics.box.ap_class_index

print("\nMÉTRICAS POR CLASE")
for metric_index, class_id in enumerate(class_indices):
    class_name = NUEVOS_NOMBRES[int(class_id)]
    print(
        f"{class_name}: Precision={class_precision[metric_index]:.4f}, "
        f"Recall={class_recall[metric_index]:.4f}, "
        f"mAP50={class_map50[metric_index]:.4f}, "
        f"mAP50-95={class_map[metric_index]:.4f}"
    )

# 13. Predicción

In [ ]:
from tkinter import Tk, filedialog

DEFAULT_IMAGE_DIR = ROOT / "tests" / "imgs"
if not DEFAULT_IMAGE_DIR.is_dir():
    DEFAULT_IMAGE_DIR = DATASET_PATH / "test" / "images"

folder_picker = Tk()
folder_picker.withdraw()
selected_folder = filedialog.askdirectory(
    title="Selecciona la carpeta de imágenes de prueba",
    initialdir=str(DEFAULT_IMAGE_DIR),
)
folder_picker.destroy()

if selected_folder:
    IMAGE_DIR = Path(selected_folder)
else:
    IMAGE_DIR = DEFAULT_IMAGE_DIR
    print("No se seleccionó carpeta; se usará:", IMAGE_DIR)

if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de imágenes seleccionada:\n{IMAGE_DIR}")

IMAGE_PATHS = sorted(
    path for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if not IMAGE_PATHS:
    raise FileNotFoundError(f"No hay imágenes compatibles en la carpeta:\n{IMAGE_DIR}")

print("Carpeta de imágenes:", IMAGE_DIR)
print("Imágenes encontradas:", len(IMAGE_PATHS))

### Ejecutar predicción

In [ ]:
if not BEST_MODEL.is_file():
    raise FileNotFoundError(f"No se encuentra best.pt para inferencia:\n{BEST_MODEL}")

PREDICTION_DEVICE = 0 if CUDA_AVAILABLE and torch.cuda.is_available() else "cpu"
PREDICTION_NAME = "AccessAI_YOLO26n_predictions_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")

results_predict = trained_model.predict(
    source=str(IMAGE_DIR),
    conf=0.25,
    imgsz=IMGSZ,
    device=PREDICTION_DEVICE,
    project=str(RUNS_DIR),
    name=PREDICTION_NAME,
    exist_ok=False,
    save=True,
    verbose=False,
)

PREDICTION_DIR = RUNS_DIR / PREDICTION_NAME
print("Dispositivo de inferencia:", PREDICTION_DEVICE)
print("Imágenes procesadas:", len(results_predict))
print("Predicciones guardadas en:", PREDICTION_DIR)
for image_result in results_predict:
    detection_count = len(image_result.boxes) if image_result.boxes is not None else 0
    print(f"{Path(image_result.path).name}: {detection_count} detecciones")

# 14. Evaluación visual

In [ ]:
if results_predict:
    first_result = results_predict[0]
    annotated_rgb = cv2.cvtColor(first_result.plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 8))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title(f"AccessAI - {Path(first_result.path).name}")
    plt.tight_layout()
    plt.show()

if RESULTS_CSV.is_file():
    training_history = pd.read_csv(RESULTS_CSV)
    training_history.columns = training_history.columns.str.strip()
    loss_columns = [
        column for column in ("train/box_loss", "train/cls_loss", "train/dfl_loss")
        if column in training_history.columns
    ]
    if loss_columns and "epoch" in training_history.columns:
        plt.figure(figsize=(12, 6))
        for column in loss_columns:
            plt.plot(training_history["epoch"], training_history[column], label=column)
        plt.xlabel("Época")
        plt.ylabel("Loss")
        plt.title("Pérdidas de entrenamiento")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    map_columns = [
        column for column in ("metrics/mAP50(B)", "metrics/mAP50-95(B)")
        if column in training_history.columns
    ]
    if map_columns and "epoch" in training_history.columns:
        plt.figure(figsize=(12, 6))
        for column in map_columns:
            plt.plot(training_history["epoch"], training_history[column], label=column)
        plt.xlabel("Época")
        plt.ylabel("mAP")
        plt.title("Métricas de validación por época")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No se encuentra results.csv:", RESULTS_CSV)

CONFUSION_MATRIX = EVAL_DIR / "confusion_matrix.png"
if CONFUSION_MATRIX.is_file():
    confusion_image = cv2.imread(str(CONFUSION_MATRIX))
    if confusion_image is None:
        raise IOError(f"No se pudo abrir la matriz de confusión:\n{CONFUSION_MATRIX}")
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(confusion_image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Matriz de confusión en test")
    plt.tight_layout()
    plt.show()
else:
    print("No se encontró la matriz de confusión:", CONFUSION_MATRIX)

# 15. Resumen final

In [ ]:
if RESULTS_CSV.is_file():
    EPOCHS_COMPLETED = len(pd.read_csv(RESULTS_CSV))
else:
    EPOCHS_COMPLETED = EPOCHS

print("=" * 60)
print("ACCESSAI - RESUMEN DEL ENTRENAMIENTO")
print("=" * 60)
print("Modelo:", "YOLO26n")
print("Clases:", ", ".join(NUEVOS_NOMBRES))
print("GPU:", GPU_NAME)
print("VRAM:", f"{GPU_VRAM_GB:.2f} GB")
print("CUDA:", f"{torch.version.cuda} (disponible={torch.cuda.is_available()})")
print("Épocas:", EPOCHS_COMPLETED)
print("Imagen:", IMAGE_DIR)
print("Batch:", BATCH)
print("Precision:", f"{PRECISION:.4f}")
print("Recall:", f"{RECALL:.4f}")
print("mAP50:", f"{MAP50:.4f}")
print("mAP50-95:", f"{MAP50_95:.4f}")
print("Best model:", BEST_MODEL)
print("Run:", RUN_DIR)
print("=" * 60)